In [ ]:
# casual forest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from econml.grf import CausalForest, RegressionForest
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12

nature_blue = '#3C5488'
grid_color = '#E5E5E5'

try:
    data = pd.read_excel("data6.xlsx")
except FileNotFoundError:
    print("错误：未找到 'data6.xlsx' 文件。")
    exit()

Y = data.iloc[:, 2].values.ravel()
W = data.iloc[:, 3].values.ravel()
X_df = data.iloc[:, 4:14].copy()

X = X_df.values

if np.isnan(X).any() or np.isnan(Y).any() or np.isnan(W).any():
    raise ValueError("数据中仍存在缺失值，请检查 data5.xlsx。")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

n_trees_list = [500, 2000, 8000, 16000]

for idx, n_trees in enumerate(n_trees_list):
    print(f"正在计算: 树的数量 = {n_trees} ...")
    np.random.seed(1)

   
    Y_forest = RegressionForest(
        n_estimators=100,
        random_state=1
    ).fit(X, Y)

    W_forest = RegressionForest(
        n_estimators=100,
        random_state=1
    ).fit(X, W)

    feature_importances = Y_forest.feature_importances_
    mean_importance = np.mean(feature_importances)

    selected_idx = np.where(feature_importances / mean_importance > 0.2)[0]
    X2 = X[:, selected_idx]

    
    tau_forest = CausalForest(
        n_estimators=n_trees,
        min_samples_leaf=5,
        random_state=1,
        inference=True
    )

    tau_forest.fit(X2, Y, W)
    tau_hat = tau_forest.predict(X2)

    
    pd.DataFrame(tau_hat, columns=['CATE']).to_excel(
        f'CausalForest_Result_{n_trees}_trees.xlsx',
        index=False
    )

    ax = axes[idx // 2, idx % 2]

    ax.hist(
        tau_hat,
        bins=35,
        color=nature_blue,
        alpha=0.85,
        edgecolor='white',
        linewidth=0.5,
        zorder=3
    )

    ax.set_xlabel('CATE', fontsize=12, labelpad=8)
    ax.set_ylabel('Frequency', fontsize=12, labelpad=10)

    ax.text(
        0.5,
        -0.12,
        f'({chr(97 + idx)}) Trees = {n_trees}',
        transform=ax.transAxes,
        fontsize=14,
        fontweight='bold',
        horizontalalignment='center',
        verticalalignment='top'
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color=grid_color, zorder=0)
    ax.tick_params(axis='both', which='major', labelsize=11, length=5)

    print(f"  -> 树数量 {n_trees} 处理完毕。")

plt.tight_layout(pad=3.0, h_pad=2.5, w_pad=3.0)

plt.savefig(
    'CausalForest_Nature_Style_BottomTitle.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
#Auto IV
import os
import random
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


DATA_PATH = "data6.xlsx"          # 支持 .xlsx / .xls / .csv
SHEET_NAME = 0
OUTPUT_XLSX = "autoiv.xlsx"

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_REP_DIM_Z = 2           
TARGET_REP_DIM_C_UNRESTRICTED = 3
TARGET_REP_DIM_C_RESTRICTED = 2

ENFORCE_C_DIM_RESTRICTION = False

EMB_DIM = 4
REP_NET_LAYERS = 2
X_NET_LAYERS = 2
EMB_NET_LAYERS = 2
Y_NET_LAYERS = 2

LR = 1e-3
DROPOUT = 0.0
EPOCHS = 1000
SIGMA = 0.1
TRAIN_RATIO = 0.6
VALID_RATIO = 0.2
TEST_RATIO = 0.2
VERBOSE_EVERY = 50

COEFS = {
    "coef_cx2y": 1.0,
    "coef_zc2x": 1.0,
    "coef_lld_zx": 1.0,
    "coef_lld_zy": 1.0,
    "coef_lld_cx": 1.0,
    "coef_lld_cy": 1.0,
    "coef_lld_zc": 1.0,
    "coef_bound_zx": 1.0,
    "coef_bound_zy": 1.0,
    "coef_bound_cx": 1.0,
    "coef_bound_cy": 1.0,
    "coef_bound_zc": 1.0,
    "coef_reg": 0.001,
}


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_table(path: str, sheet_name=0) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name)
    if ext == ".csv":
        return pd.read_csv(path)
    raise ValueError("DATA_PATH 仅支持 .xlsx / .xls / .csv")


def to_tensor(arr: np.ndarray) -> torch.Tensor:
    return torch.tensor(arr, dtype=torch.float32, device=DEVICE)


def standardize_by_train(train_arr: np.ndarray, full_arr: np.ndarray) -> Tuple[np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    scaler.fit(train_arr)
    return scaler.transform(full_arr), scaler


def safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def decide_rep_dims(
    n_candidate_v: int,
    target_z: int = 2,
    c_unrestricted: int = 3,
    c_restricted: int = 2,
    enforce_restriction: bool = False,
) -> Tuple[int, int, str]:
    """
    按你的要求做显式判断：
    1. 若 enforce_restriction=True，则使用受限逻辑：
       rep_dim_c <= n_candidate_v - rep_dim_z
    2. 若 enforce_restriction=False，则按 AutoIV 论文/原仓库口径，直接取 c_unrestricted
    """
    if target_z <= 0:
        raise ValueError("自动工具变量个数必须为正数。")
    if n_candidate_v <= 0:
        raise ValueError("候选工具变量个数必须为正数。")
    if target_z > n_candidate_v:
        raise ValueError("自动工具变量个数不能超过候选工具变量个数。")

    if enforce_restriction:
        max_c = max(1, n_candidate_v - target_z)
        rep_dim_c = min(c_restricted, max_c)
        rule_text = (
            f"采用受限逻辑：rep_dim_c <= 候选工具变量数 - 自动工具变量数，"
            f"因此 rep_dim_z={target_z}, rep_dim_c={rep_dim_c}"
        )
    else:
        rep_dim_c = c_unrestricted
        rule_text = (
            f"采用 AutoIV 论文/原仓库口径：不对 rep_dim_c 施加 "
            f"候选工具变量数 - 自动工具变量数 的硬约束，"
            f"因此 rep_dim_z={target_z}, rep_dim_c={rep_dim_c}"
        )

    return target_z, rep_dim_c, rule_text


class MLP(nn.Module):
    def __init__(self, dims: List[int], dropout: float = 0.0, last_activation: bool = False):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            in_dim = dims[i]
            out_dim = dims[i + 1]
            layers.append(nn.Linear(in_dim, out_dim))
            is_last = (i == len(dims) - 2)
            if (not is_last) or last_activation:
                layers.append(nn.ELU())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MINet(nn.Module):
    def __init__(self, dim_in: int, dim_out: int):
        super().__init__()
        hidden = max(1, dim_in // 2)

        self.mu_net = nn.Sequential(
            nn.Linear(dim_in, hidden),
            nn.ELU(),
            nn.Linear(hidden, dim_out),
        )

        self.logvar_net = nn.Sequential(
            nn.Linear(dim_in, hidden),
            nn.ELU(),
            nn.Linear(hidden, dim_out),
            nn.Tanh(),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(
        self,
        inp: torch.Tensor,
        outp: torch.Tensor,
        mi_mode: str,
        x_for_cond: torch.Tensor = None,
        sigma: float = 0.1,
    ) -> Dict[str, torch.Tensor]:
        n = inp.shape[0]
        mu = self.mu_net(inp)
        logvar = self.logvar_net(inp)

        perm = torch.randperm(n, device=inp.device)
        outp_rand = outp[perm]

        lld = -torch.mean(
            torch.sum(-((outp - mu) ** 2) / torch.exp(logvar) - logvar, dim=1)
        )

        pos = -((mu - outp) ** 2) / torch.exp(logvar)
        neg = -((mu - outp_rand) ** 2) / torch.exp(logvar)

        if x_for_cond is not None:
            x_rand = x_for_cond[perm]
            w = torch.exp(-((x_for_cond - x_rand) ** 2) / (2 * sigma ** 2))
            w_soft = torch.softmax(w, dim=0)
        else:
            w_soft = torch.full_like(pos, 1.0 / n)

        if mi_mode == "min":
            pn = 1.0
        elif mi_mode == "max":
            pn = -1.0
        else:
            raise ValueError("mi_mode 只能是 'min' 或 'max'")

        bound = pn * torch.sum(w_soft * (pos - neg))

        return {
            "mu": mu,
            "logvar": logvar,
            "lld": lld,
            "bound": bound,
        }


class AutoIV(nn.Module):
    def __init__(
        self,
        dim_x: int,
        dim_v: int,
        dim_y: int,
        rep_dim_z: int = 2,
        rep_dim_c: int = 3,
        emb_dim: int = 4,
        rep_layers: int = 2,
        x_layers: int = 2,
        emb_layers: int = 2,
        y_layers: int = 2,
        dropout: float = 0.0,
        sigma: float = 0.1,
        coefs: Dict[str, float] = None,
    ):
        super().__init__()
        self.dim_x = dim_x
        self.dim_v = dim_v
        self.dim_y = dim_y
        self.rep_dim_z = rep_dim_z
        self.rep_dim_c = rep_dim_c
        self.emb_dim = emb_dim
        self.dropout = dropout
        self.sigma = sigma
        self.coefs = coefs if coefs is not None else COEFS

        rep_z_dims = [dim_v] + [rep_dim_z] * rep_layers
        rep_c_dims = [dim_v] + [rep_dim_c] * rep_layers
        self.rep_z_net = MLP(rep_z_dims, dropout=dropout, last_activation=False)
        self.rep_c_net = MLP(rep_c_dims, dropout=dropout, last_activation=False)

        x_input_dim = rep_dim_z + rep_dim_c
        x_dims = self._shrinking_dims(x_input_dim, dim_x, x_layers)
        self.x_net = MLP(x_dims, dropout=dropout, last_activation=False)

        emb_dims = [dim_x] + [emb_dim] * emb_layers
        self.emb_net = MLP(emb_dims, dropout=dropout, last_activation=False)

        y_input_dim = rep_dim_c + emb_dim
        y_dims = self._shrinking_dims(y_input_dim, dim_y, y_layers)
        self.y_net = MLP(y_dims, dropout=dropout, last_activation=False)

        self.mi_zx = MINet(rep_dim_z, dim_x)
        self.mi_zy = MINet(rep_dim_z, dim_y)
        self.mi_cx = MINet(rep_dim_c, dim_x)
        self.mi_cy = MINet(rep_dim_c, dim_y)
        self.mi_zc = MINet(rep_dim_z, rep_dim_c)

    @staticmethod
    def _shrinking_dims(dim_in: int, dim_out: int, n_layers: int) -> List[int]:
        dims = [dim_in]
        for i in range(n_layers):
            if i == n_layers - 1:
                dims.append(dim_out)
            else:
                next_dim = max(1, dim_in // ((i + 1) * 2))
                dims.append(next_dim)
        return dims

    def forward(self, v: torch.Tensor, x: torch.Tensor, y: torch.Tensor) -> Dict[str, torch.Tensor]:
        z = self.rep_z_net(v)
        c = self.rep_c_net(v)

        zc = torch.cat([z, c], dim=1)
        x_pre = self.x_net(zc)

        x_emb = self.emb_net(x_pre)
        cx = torch.cat([c, x_emb], dim=1)
        y_pre = self.y_net(cx)

        zx = self.mi_zx(z, x, mi_mode="max")
        zy = self.mi_zy(z, y, mi_mode="min", x_for_cond=x, sigma=self.sigma)
        cx_mi = self.mi_cx(c, x, mi_mode="max")
        cy = self.mi_cy(c, y, mi_mode="max")
        zc_mi = self.mi_zc(z, c, mi_mode="min")

        return {
            "z": z,
            "c": c,
            "x_pre": x_pre,
            "x_emb": x_emb,
            "y_pre": y_pre,
            "zx": zx,
            "zy": zy,
            "cx": cx_mi,
            "cy": cy,
            "zc": zc_mi,
        }

    def reg_loss(self) -> torch.Tensor:
        total = torch.tensor(0.0, device=DEVICE)
        nets = [self.rep_z_net, self.rep_c_net, self.emb_net, self.x_net, self.y_net]
        for net in nets:
            for p in net.parameters():
                total = total + torch.sum(p ** 2) / 2.0
        return total / len(nets)

    def get_losses(self, out: Dict[str, torch.Tensor], x: torch.Tensor, y: torch.Tensor) -> Dict[str, torch.Tensor]:
        loss_cx2y = torch.mean((y - out["y_pre"]) ** 2)
        loss_zc2x = torch.mean((x - out["x_pre"]) ** 2)
        loss_reg = self.reg_loss()

        loss_lld = (
            self.coefs["coef_lld_zy"] * out["zy"]["lld"]
            + self.coefs["coef_lld_cx"] * out["cx"]["lld"]
            + self.coefs["coef_lld_zx"] * out["zx"]["lld"]
            + self.coefs["coef_lld_cy"] * out["cy"]["lld"]
            + self.coefs["coef_lld_zc"] * out["zc"]["lld"]
        )

        loss_bound = (
            self.coefs["coef_bound_zy"] * out["zy"]["bound"]
            + self.coefs["coef_bound_cx"] * out["cx"]["bound"]
            + self.coefs["coef_bound_zx"] * out["zx"]["bound"]
            + self.coefs["coef_bound_cy"] * out["cy"]["bound"]
            + self.coefs["coef_bound_zc"] * out["zc"]["bound"]
            + self.coefs["coef_reg"] * loss_reg
        )

        loss_2stage = (
            self.coefs["coef_cx2y"] * loss_cx2y
            + self.coefs["coef_zc2x"] * loss_zc2x
            + self.coefs["coef_reg"] * loss_reg
        )

        return {
            "loss_cx2y": loss_cx2y,
            "loss_zc2x": loss_zc2x,
            "loss_reg": loss_reg,
            "loss_lld": loss_lld,
            "loss_bound": loss_bound,
            "loss_2stage": loss_2stage,
        }


@dataclass
class PreparedData:
    df_raw: pd.DataFrame
    fit_df: pd.DataFrame

    year_col: str
    county_col: str
    y_col: str
    x_col: str
    v_cols: List[str]

    fit_row_index: np.ndarray

    x_all_scaled: np.ndarray
    y_all_scaled: np.ndarray
    v_all_scaled: np.ndarray

    x_scaler: StandardScaler
    y_scaler: StandardScaler
    v_scaler: StandardScaler

    train_idx: np.ndarray
    valid_idx: np.ndarray
    test_idx: np.ndarray

    rep_dim_z: int
    rep_dim_c: int
    rule_text: str


def prepare_data(df_raw: pd.DataFrame) -> PreparedData:
    df = df_raw.copy()

    year_col = df.columns[0]
    county_col = df.columns[1]
    y_col = df.columns[2]
    x_col = df.columns[3]
    v_cols = df.columns[14:18].tolist()

    df[y_col] = safe_numeric(df[y_col])
    df[x_col] = safe_numeric(df[x_col])
    for col in v_cols:
        df[col] = safe_numeric(df[col])

    rep_dim_z, rep_dim_c, rule_text = decide_rep_dims(
        n_candidate_v=len(v_cols),
        target_z=TARGET_REP_DIM_Z,
        c_unrestricted=TARGET_REP_DIM_C_UNRESTRICTED,
        c_restricted=TARGET_REP_DIM_C_RESTRICTED,
        enforce_restriction=ENFORCE_C_DIM_RESTRICTION,
    )

    fit_cols = [y_col, x_col] + v_cols
    fit_mask = df[fit_cols].notna().all(axis=1)
    fit_df = df.loc[fit_mask].copy()
    fit_row_index = fit_df.index.to_numpy()

    if fit_df.shape[0] < 30:
        raise ValueError("y、x 和候选工具变量同时非缺失的样本过少，无法训练 AutoIV。")

    x_all = fit_df[[x_col]].astype(float).values
    y_all = fit_df[[y_col]].astype(float).values
    v_all = fit_df[v_cols].astype(float).values

    n = fit_df.shape[0]
    all_idx = np.arange(n)
    train_idx, temp_idx = train_test_split(all_idx, train_size=TRAIN_RATIO, random_state=SEED)
    valid_share = VALID_RATIO / (VALID_RATIO + TEST_RATIO)
    valid_idx, test_idx = train_test_split(temp_idx, train_size=valid_share, random_state=SEED)

    x_all_scaled, x_scaler = standardize_by_train(x_all[train_idx], x_all)
    y_all_scaled, y_scaler = standardize_by_train(y_all[train_idx], y_all)
    v_all_scaled, v_scaler = standardize_by_train(v_all[train_idx], v_all)

    return PreparedData(
        df_raw=df_raw.copy(),
        fit_df=fit_df.copy(),
        year_col=year_col,
        county_col=county_col,
        y_col=y_col,
        x_col=x_col,
        v_cols=v_cols,
        fit_row_index=fit_row_index,
        x_all_scaled=x_all_scaled,
        y_all_scaled=y_all_scaled,
        v_all_scaled=v_all_scaled,
        x_scaler=x_scaler,
        y_scaler=y_scaler,
        v_scaler=v_scaler,
        train_idx=train_idx,
        valid_idx=valid_idx,
        test_idx=test_idx,
        rep_dim_z=rep_dim_z,
        rep_dim_c=rep_dim_c,
        rule_text=rule_text,
    )


def evaluate_split(model: AutoIV, pdata: PreparedData, idx: np.ndarray) -> Dict[str, float]:
    model.eval()
    with torch.no_grad():
        v = to_tensor(pdata.v_all_scaled[idx])
        x = to_tensor(pdata.x_all_scaled[idx])
        y = to_tensor(pdata.y_all_scaled[idx])
        out = model(v, x, y)
        losses = model.get_losses(out, x, y)
        return {
            "mse_y": float(losses["loss_cx2y"].detach().cpu().item()),
            "mse_x": float(losses["loss_zc2x"].detach().cpu().item()),
            "loss_lld": float(losses["loss_lld"].detach().cpu().item()),
            "loss_bound": float(losses["loss_bound"].detach().cpu().item()),
            "loss_2stage": float(losses["loss_2stage"].detach().cpu().item()),
        }


def train_autoiv(pdata: PreparedData) -> AutoIV:
    model = AutoIV(
        dim_x=1,
        dim_v=len(pdata.v_cols),
        dim_y=1,
        rep_dim_z=pdata.rep_dim_z,
        rep_dim_c=pdata.rep_dim_c,
        emb_dim=EMB_DIM,
        rep_layers=REP_NET_LAYERS,
        x_layers=X_NET_LAYERS,
        emb_layers=EMB_NET_LAYERS,
        y_layers=Y_NET_LAYERS,
        dropout=DROPOUT,
        sigma=SIGMA,
        coefs=COEFS,
    ).to(DEVICE)

    lld_params = (
        list(model.mi_zx.parameters())
        + list(model.mi_zy.parameters())
        + list(model.mi_cx.parameters())
        + list(model.mi_cy.parameters())
        + list(model.mi_zc.parameters())
    )

    rep_params = list(model.rep_z_net.parameters()) + list(model.rep_c_net.parameters())

    stage2_params = (
        rep_params
        + list(model.x_net.parameters())
        + list(model.emb_net.parameters())
        + list(model.y_net.parameters())
    )

    opt_lld = optim.Adam(lld_params, lr=LR)
    opt_bound = optim.Adam(rep_params, lr=LR)
    opt_2stage = optim.Adam(stage2_params, lr=LR)

    x_train = to_tensor(pdata.x_all_scaled[pdata.train_idx])
    y_train = to_tensor(pdata.y_all_scaled[pdata.train_idx])
    v_train = to_tensor(pdata.v_all_scaled[pdata.train_idx])

    for epoch in range(1, EPOCHS + 1):
        model.train()

        opt_lld.zero_grad()
        out = model(v_train, x_train, y_train)
        losses = model.get_losses(out, x_train, y_train)
        losses["loss_lld"].backward()
        opt_lld.step()

        opt_bound.zero_grad()
        out = model(v_train, x_train, y_train)
        losses = model.get_losses(out, x_train, y_train)
        losses["loss_bound"].backward()
        opt_bound.step()

        opt_2stage.zero_grad()
        out = model(v_train, x_train, y_train)
        losses = model.get_losses(out, x_train, y_train)
        losses["loss_2stage"].backward()
        opt_2stage.step()

        if epoch == 1 or epoch % VERBOSE_EVERY == 0 or epoch == EPOCHS:
            tr = evaluate_split(model, pdata, pdata.train_idx)
            va = evaluate_split(model, pdata, pdata.valid_idx)
            te = evaluate_split(model, pdata, pdata.test_idx)
            print(
                f"Epoch {epoch:04d} | "
                f"Train mse_y={tr['mse_y']:.6f}, mse_x={tr['mse_x']:.6f} | "
                f"Valid mse_y={va['mse_y']:.6f}, mse_x={va['mse_x']:.6f} | "
                f"Test mse_y={te['mse_y']:.6f}, mse_x={te['mse_x']:.6f}"
            )

    return model

def extract_representations(model: AutoIV, pdata: PreparedData) -> pd.DataFrame:
    model.eval()
    with torch.no_grad():
        v_all = to_tensor(pdata.v_all_scaled)
        x_all = to_tensor(pdata.x_all_scaled)
        y_all = to_tensor(pdata.y_all_scaled)
        out = model(v_all, x_all, y_all)

        z = out["z"].detach().cpu().numpy()
        c = out["c"].detach().cpu().numpy()

    full_df = pdata.df_raw.copy()

    # 先创建自动工具变量列
    for j in range(pdata.rep_dim_z):
        full_df[f"auto_iv{j + 1}"] = np.nan

    # 再创建自动控制变量列
    for j in range(pdata.rep_dim_c):
        full_df[f"auto_c{j + 1}"] = np.nan

    # 写回有效样本对应的表示
    for j in range(pdata.rep_dim_z):
        full_df.loc[pdata.fit_row_index, f"auto_iv{j + 1}"] = z[:, j]

    for j in range(pdata.rep_dim_c):
        full_df.loc[pdata.fit_row_index, f"auto_c{j + 1}"] = c[:, j]

    return full_df


def main():
    set_seed(SEED)
    print(f"Using device: {DEVICE}")

    df_raw = read_table(DATA_PATH, SHEET_NAME)
    pdata = prepare_data(df_raw)

    print("=" * 100)
    print("列识别结果：")
    print(f"year:   {pdata.year_col}")
    print(f"county: {pdata.county_col}")
    print(f"y:      {pdata.y_col}")
    print(f"x:      {pdata.x_col}")
    print(f"V cols: {pdata.v_cols}")
    print("=" * 100)
    print("维度设定：")
    print(pdata.rule_text)
    print("=" * 100)

    print(f"原始总样本数：{pdata.df_raw.shape[0]}")
    print(f"进入 AutoIV 的有效样本数：{pdata.fit_df.shape[0]}")
    print(f"Train / Valid / Test = {len(pdata.train_idx)} / {len(pdata.valid_idx)} / {len(pdata.test_idx)}")
    print("=" * 100)

    model = train_autoiv(pdata)
    output_df = extract_representations(model, pdata)

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        output_df.to_excel(writer, index=False)

    print("\n" + "=" * 100)
    print(f"结果文件已保存：{OUTPUT_XLSX}")

    added_cols = [f"auto_iv{i + 1}" for i in range(pdata.rep_dim_z)] + \
                 [f"auto_c{i + 1}" for i in range(pdata.rep_dim_c)]
    print("输出内容为原始全部数据，并在最后额外新增以下列：")
    print(", ".join(added_cols))
    print("=" * 100)


if __name__ == "__main__":
    main()